# Credit Card Fraud Detection - Analysis Notebook

This project applies a complete data analysis pipeline to a credit-card transaction dataset. The project track tells us that the dataset is about fraud detection, but the notebook does not start from the final conclusion. Instead, it follows the same practical logic we would use as students working through the problem step by step:

1. load the data and check what is actually inside it;
2. understand the target variable and the meaning of the class labels;
3. inspect data quality, imbalance, and important numerical distributions;
4. prepare the features without leaking information from the test set;
5. compare multiple classification models;
6. interpret the results using metrics that match the business problem.

The general question is:

> **Can we build a model that helps identify suspicious transactions better than a naive rule, while keeping the trade-off between missed frauds and false alarms understandable?**

At the beginning we do not assume that accuracy is the right metric. We first inspect the target distribution. Only after seeing the class imbalance do we decide to focus on precision, recall, F1-score, ROC-AUC, and especially PR-AUC.


## 1. Setup

We start with the technical setup. All libraries are imported in one cell so that the rest of the notebook is easier to follow.

The random seed is fixed because some operations, such as the train/test split and randomized hyperparameter search, contain randomness. Fixing the seed makes the results reproducible.


In [1]:
# Utilities for file and folder path handling
from pathlib import Path

# Used to suppress non-critical warning messages
import warnings

# Numerical computations and array manipulation
import numpy as np

# Data loading, cleaning, and analysis
import pandas as pd

# Interactive data visualizations
import plotly.express as px

# Display pandas DataFrames nicely in Jupyter notebooks
from IPython.display import display

# Specific warning generated when machine learning models fail to converge
from sklearn.exceptions import ConvergenceWarning

# Tools for model evaluation, validation, and dataset splitting
from sklearn.model_selection import (
    GridSearchCV,          # Exhaustive hyperparameter search
    RandomizedSearchCV,    # Random hyperparameter search
    StratifiedKFold,       # Stratified cross-validation
    train_test_split,      # Split data into training and testing sets
)

# Robust feature scaling, less sensitive to outliers
from sklearn.preprocessing import RobustScaler

# Logistic Regression classification model
from sklearn.linear_model import LogisticRegression

# Random Forest classification model
from sklearn.ensemble import RandomForestClassifier

# Performance evaluation metrics
from sklearn.metrics import (
    average_precision_score,  # Area under Precision-Recall curve
    f1_score,                 # Harmonic mean of precision and recall
    precision_score,          # Precision metric
    recall_score,             # Recall metric
    roc_auc_score,            # Area under ROC curve
)

# Import XGBoost classifier if available
try:
    from xgboost import XGBClassifier
except ImportError as exc:
    raise ImportError(
        "XGBoost is required for section 5.3. Install it with: pip install xgboost"
    ) from exc

# Fixed random seed for reproducibility
RANDOM_STATE = 42

# Path to the dataset file
DATA_PATH = Path("creditcard.csv")

# Display settings for pandas DataFrames
pd.set_option("display.max_columns", 40)  # Show up to 40 columns
pd.set_option("display.float_format", lambda x: f"{x:.6f}")  # Show 6 decimal places

# Hide non-critical warnings to keep output cleaner
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=ConvergenceWarning)

## 2. Loading the data

The first practical step is simply to load the file and make sure the notebook can find it.

At this point we only know that the project track is about credit-card fraud detection and that the dataset should contain a target column. We still need to inspect the table before deciding how to prepare and evaluate the models.


In [2]:
# Check whether the dataset file exists in the specified location
if not DATA_PATH.exists():
    # Raise an error if the file cannot be found
    raise FileNotFoundError(
        f"{DATA_PATH} not found. Place creditcard.csv in the same folder as this notebook."
    )

# Load the credit card transactions dataset into a pandas DataFrame
df = pd.read_csv(DATA_PATH)

# Confirm that the dataset has been successfully loaded
print(f"Loaded dataset: {DATA_PATH}")

Loaded dataset: creditcard.csv


## 3. First inspection of the dataset

Before building any model, we ask basic questions about the data:

- How many rows and columns are there?
- Which columns are available?
- Is the target column really present?
- Which values does the target column contain?
- Are there missing values?
- Is one class much rarer than the other?
- What do the directly interpretable variables, especially `Time` and `Amount`, look like?

This is the first checkpoint of the analysis. If something is wrong here, every modelling result later would be unreliable.


In [3]:
# Display the dimensions of the dataset (rows and columns)
print("Dataset shape (rows, cols):", df.shape)

# Display the total number of features/columns
print("Number of columns:", df.shape[1])

# Print the names of all columns in the dataset
print("\nColumns:")
print(df.columns.to_list())

# Verify that the target variable 'Class' is present
assert "Class" in df.columns, "The target column 'Class' is missing."

# Extract and display the unique values of the target variable
observed_classes = sorted(df["Class"].dropna().unique().tolist())
print("\nObserved values in target column 'Class':", observed_classes)

# Ensure that the classification problem is binary (0 = legitimate, 1 = fraud)
assert observed_classes == [0, 1], "Expected a binary target encoded as 0 and 1."

# Define the interpretation of the target labels
# 0 = legitimate transaction, 1 = fraudulent transaction
CLASS_LABELS = {
    0: "Legitimate (0)",
    1: "Fraud (1)",
}
print("Working label interpretation:", CLASS_LABELS)

# Display the first five records of the dataset
display(df.head())

# Count missing values in each column
null_counts = df.isna().sum()

# Calculate the total number of missing values
null_total = int(null_counts.sum())
print("\nTotal null values in dataset:", null_total)

# Report whether missing values are present
if null_total == 0:
    print("No missing values detected.")
else:
    # Display only columns containing missing values
    display(null_counts[null_counts > 0].sort_values(ascending=False))

# Calculate the number of samples in each class
class_counts = df["Class"].value_counts().sort_index()

# Calculate the relative frequency of each class
class_rates = class_counts / len(df)

print("\nClass distribution (counts):")
print(class_counts)

print("\nClass distribution (rates):")
print((class_rates * 100).map(lambda x: f"{x:.6f}%"))

# Calculate the prevalence of fraudulent transactions
fraud_rate = float(class_rates.loc[1])

# Calculate the baseline accuracy obtained by always predicting the majority class
baseline_accuracy = float(class_rates.loc[0])

print(f"\nFraud prevalence: {fraud_rate:.6%}")
print(f"Majority-class baseline accuracy (always legitimate): {baseline_accuracy:.6%}")

# Generate descriptive statistics for the Time and Amount features
for col in ["Time", "Amount"]:
    print(f"\n{col} summary:")
    display(df[col].describe())

Dataset shape (rows, cols): (284807, 31)
Number of columns: 31

Columns:
['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount', 'Class']

Observed values in target column 'Class': [0, 1]
Working label interpretation: {0: 'Legitimate (0)', 1: 'Fraud (1)'}


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11,V12,V13,V14,V15,V16,V17,V18,V19,V20,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.000000,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,0.090794,-0.551600,-0.617801,-0.991390,-0.311169,1.468177,-0.470401,0.207971,0.025791,0.403993,0.251412,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.620000,0
1,0.000000,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,-0.166974,1.612727,1.065235,0.489095,-0.143772,0.635558,0.463917,-0.114805,-0.183361,-0.145783,-0.069083,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.690000,0
2,1.000000,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,0.207643,0.624501,0.066084,0.717293,-0.165946,2.345865,-2.890083,1.109969,-0.121359,-2.261857,0.524980,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.660000,0
3,1.000000,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,-0.054952,-0.226487,0.178228,0.507757,-0.287924,-0.631418,-1.059647,-0.684093,1.965775,-1.232622,-0.208038,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.500000,0
4,2.000000,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,0.753074,-0.822843,0.538196,1.345852,-1.119670,0.175121,-0.451449,-0.237033,-0.038195,0.803487,0.408542,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.990000,0



Total null values in dataset: 0
No missing values detected.

Class distribution (counts):
Class
0    284315
1       492
Name: count, dtype: int64

Class distribution (rates):
Class
0    99.827251%
1     0.172749%
Name: count, dtype: object

Fraud prevalence: 0.172749%
Majority-class baseline accuracy (always legitimate): 99.827251%

Time summary:


count   284807.000000
mean     94813.859575
std      47488.145955
min          0.000000
25%      54201.500000
50%      84692.000000
75%     139320.500000
max     172792.000000
Name: Time, dtype: float64


Amount summary:


count   284807.000000
mean        88.349619
std        250.120109
min          0.000000
25%          5.600000
50%         22.000000
75%         77.165000
max      25691.160000
Name: Amount, dtype: float64

### What we learn from the first inspection

The file is loaded correctly: it contains **284,807 transactions** and **31 columns**.

The target column `Class` is present and has only two observed values: `0` and `1`. Using the dataset convention for this project, we interpret `0` as a legitimate transaction and `1` as a fraudulent transaction. We do not use this as a modelling result; we still inspect the counts to understand the difficulty of the task.

The class distribution shows the main problem: there are only **492 fraud transactions**, about **0.173%** of the dataset. This means a model that always predicts the majority class would already obtain about **99.83% accuracy**, but it would detect no fraud. From this point onward, accuracy alone is not enough.

There are no missing values, so no imputation is required. `Amount` is strongly skewed: the median is **22.00**, while the maximum is **25,691.16**. This suggests that scaling should be robust to extreme values.


## 4. Exploratory Data Analysis

Now that the dataset structure is clear, we use visualizations and summary statistics to answer more specific questions:

1. **How severe is the class imbalance visually?**
2. **Does `Amount` behave differently for legitimate and fraudulent transactions?**
3. **Are some anonymised variables more linearly related to the target than others?**
4. **Are there data-quality details, such as duplicated rows or zero-amount transactions, that we should know before modelling?**

The aim is not to force a story from anonymised variables, but to make informed modelling choices.


In [4]:
# Compute class distribution (counts) from the target variable
class_plot = (
    df["Class"]
    .value_counts()
    .rename_axis("Class")  # turn index name into a column
    .reset_index(name="Count")  # convert Series into a DataFrame
)

# Map numeric class labels to human-readable labels
class_plot["Label"] = class_plot["Class"].map(CLASS_LABELS)

# Create an interactive bar chart showing class imbalance
fig = px.bar(
    class_plot,
    x="Label",  # x-axis shows class names (Legitimate / Fraud)
    y="Count",  # y-axis shows number of samples per class
    text="Count",  # display counts on top of bars
    title="Class distribution: legitimate transactions vs fraud",
)

# Improve bar text formatting and hover information
fig.update_traces(
    texttemplate="%{text:,}",  # format numbers with thousand separators
    textposition="outside",    # place labels above bars
    hovertemplate="%{x}<br>Count = %{y:,}<extra></extra>",
)

# Set axis labels and remove legend (not needed for single series)
fig.update_layout(
    xaxis_title="Class",
    yaxis_title="Count",
    showlegend=False
)

# Render the interactive plot
fig.show()

### Answer from the class plot

The plot confirms visually what the counts already suggested: this is a rare-event classification problem. The positive class is so small that a normal accuracy score would be misleading.

This changes the way we evaluate the models. We need metrics that focus on the positive class:

- **recall**, because a false negative means a real fraud is missed;
- **precision**, because too many false positives create unnecessary checks and customer friction;
- **F1-score**, because it summarizes the precision-recall balance at a selected threshold;
- **PR-AUC**, because it evaluates ranking quality when positives are rare.

ROC-AUC is still reported, but PR-AUC is more informative for the final comparison.


In [5]:
# ECDF (Empirical Cumulative Distribution Function) of transaction Amount grouped by class.
# A logarithmic x-axis is used because the Amount feature is highly right-skewed.
# Zero values are excluded only for visualization purposes since log(0) is undefined.
# They are still included in all numerical summaries below.

# Filter dataset to remove zero amounts (required for log scale visualization)
plot_df = df.loc[df["Amount"] > 0, ["Amount", "Class"]].copy()

# Replace numeric class labels with readable labels for plotting
plot_df["Class"] = plot_df["Class"].map(CLASS_LABELS)

# Create ECDF plot with class-based coloring
fig = px.ecdf(
    plot_df,
    x="Amount",      # transaction amount
    color="Class",   # separate ECDF per class
    log_x=True,      # logarithmic scale due to skewness
    marginal="box",  # add boxplots for distribution summary
    title="Amount distribution by class — ECDF on log scale",
)

# Customize axis labels and legend
fig.update_layout(
    xaxis_title="Amount (log scale)",
    yaxis_title="Cumulative share within class",
    legend_title="Class",
)

# Display interactive figure
fig.show()

# Compute summary statistics for transaction amounts by class
amount_summary = pd.DataFrame(
    {
        "median": [
            df.loc[df["Class"] == 0, "Amount"].median(),
            df.loc[df["Class"] == 1, "Amount"].median(),
        ],
        "p90": [
            df.loc[df["Class"] == 0, "Amount"].quantile(0.90),
            df.loc[df["Class"] == 1, "Amount"].quantile(0.90),
        ],
        "p99": [
            df.loc[df["Class"] == 0, "Amount"].quantile(0.99),
            df.loc[df["Class"] == 1, "Amount"].quantile(0.99),
        ],
    },
    index=["Legitimate", "Fraud"],
)

# Display the summary table
display(amount_summary)

,median,p90,p99
Legitimate,22.000000,202.724000,1016.966400
Fraud,9.250000,346.746000,1357.427900


### Answer from the `Amount` distribution

`Amount` is one of the few interpretable columns in the dataset, so it deserves a separate check.

The ECDF is used instead of a simple histogram because the two classes have very different sizes. A histogram would mostly show the majority class, while the ECDF lets us compare the shapes of the distributions more clearly.

This step does not prove that `Amount` alone can detect fraud. It only tells us that `Amount` is skewed and should be transformed carefully before using models that are sensitive to scale, such as Logistic Regression.


In [6]:
# Compute the correlation matrix for all numeric features in the dataset
corr = df.corr(numeric_only=True)

# Visualize the correlation matrix using an interactive heatmap
fig = px.imshow(
    corr,
    color_continuous_scale="RdBu_r",  # diverging color scale (negative to positive correlation)
    zmin=-1,                          # minimum correlation value
    zmax=1,                           # maximum correlation value
    aspect="auto",                    # allow rectangular heatmap cells
    title="Correlation heatmap including the target variable",
)

# Place x-axis labels at the bottom for readability
fig.update_xaxes(side="bottom")

# Adjust figure size and colorbar label
fig.update_layout(
    width=900,
    height=750,
    coloraxis_colorbar_title="corr"
)

# Display the heatmap
fig.show()

# Extract and rank correlations with the target variable (Class)
if "Class" in corr.columns:
    top_corr = (
        corr["Class"]              # correlation with target
        .drop("Class")             # remove self-correlation
        .abs()                     # take absolute value for strength ranking
        .sort_values(ascending=False)
        .head(10)                  # top 10 most correlated features
    )

    print("Top absolute correlations with Class (linear):")
    display(top_corr.to_frame(name="|corr(Class)|"))

Top absolute correlations with Class (linear):


,|corr(Class)|
V17,0.326481
V14,0.302544
V12,0.260593
V10,0.216883
V16,0.196539
V3,0.192961
V7,0.187257
V11,0.154876
V4,0.133447
V18,0.111485


### Answer from the correlation check

The correlation table shows which variables have the strongest **linear** relationship with `Class`. Variables such as **V17**, **V14**, **V12**, and **V10** appear near the top.

This is useful, but it must be interpreted carefully. Correlation does not prove causation, and it only captures linear relationships. Since the dataset contains anonymised PCA components, we cannot give a direct business meaning to these variables.

What we can conclude is more modest: some anonymised components contain signal related to the target, and non-linear models may be able to combine these signals better than a purely linear model.


## 5. Data preparation

After the exploratory checks, we prepare the data for modelling.

The preparation choices are based on what we observed:

1. `Class` is the target, and all other columns are features;
2. the class distribution is highly imbalanced, so the split must be stratified;
3. the test set must remain untouched until evaluation;
4. `Time` and `Amount` have a different scale from the PCA variables, so they are scaled;
5. the scaler is fitted only on the training set to avoid data leakage.


In [7]:
# Count fully duplicated rows in the dataset
duplicate_rows = int(df.duplicated().sum())
print("Fully duplicated rows:", duplicate_rows)

# Create a mask identifying transactions with zero amount
zero_amount_mask = df["Amount"] == 0

# Build a summary comparing zero-amount vs non-zero-amount transactions
amount_zero_summary = (
    df.assign(_zero=zero_amount_mask)  # temporary flag column for grouping
    .groupby("_zero")["Class"]         # group by zero vs non-zero amount
    .agg(
        n="count",        # total number of transactions
        frauds="sum",     # number of fraud cases (since Class=1 is fraud)
        fraud_rate="mean" # proportion of fraud in each group
    )
    .rename(index={False: "Amount > 0", True: "Amount == 0"})  # readable labels
)

# Display duplicate count
print("\nFully duplicated rows:", duplicate_rows)

# Display grouped summary statistics
print("\nSummary by Amount zero vs positive:")
print(amount_zero_summary.to_string())

Fully duplicated rows: 1081

Fully duplicated rows: 1081

Summary by Amount zero vs positive:
                  n  frauds  fraud_rate
_zero                                  
Amount > 0   282982     465    0.001643
Amount == 0    1825      27    0.014795


### Data-quality notes

There are **1,081 fully duplicated rows**. We count them because duplicates can matter, but we do not remove them in this version of the project. The reason is methodological: the objective is to compare models on the same input data and avoid changing the benchmark halfway through the notebook.

The zero-amount transactions are also checked. The group with `Amount == 0` has a higher fraud rate than the positive-amount group. We do not remove these rows, because they may contain useful information rather than being simple errors.


In [8]:
# Define feature columns by excluding the target variable
FEATURE_COLS = [col for col in df.columns if col != "Class"]

# Split dataset into features (X) and target (y)
X = df[FEATURE_COLS].copy()
y = df["Class"].copy()

# Split data into training and testing sets
# stratify=y ensures the same class distribution in both sets (important for imbalanced data)
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,          # 20% of data used for testing
    stratify=y,              # preserve class imbalance ratio
    random_state=RANDOM_STATE # ensure reproducibility
)

# Display dataset split shapes
print("Train shape:", X_train.shape, "| Test shape:", X_test.shape)

# Show class distribution in training set
print("\nTrain class counts:\n", y_train.value_counts().sort_index())

# Show class distribution in test set
print("\nTest class counts:\n", y_test.value_counts().sort_index())

# Compute and display fraud rate in training set
print("\nTrain fraud rate (mean of Class):", float(y_train.mean()))

# Compute and display fraud rate in test set
print("Test fraud rate (mean of Class):", float(y_test.mean()))

Train shape: (227845, 30) | Test shape: (56962, 30)

Train class counts:
 Class
0    227451
1       394
Name: count, dtype: int64

Test class counts:
 Class
0    56864
1       98
Name: count, dtype: int64

Train fraud rate (mean of Class): 0.001729245759178389
Test fraud rate (mean of Class): 0.0017204452090867595


### Why we use a stratified split

The train and test fraud rates are almost identical. This is important because fraud cases are very rare: without stratification, a random split could accidentally create a test set with too few or too many fraud cases.

The test set is kept separate and is not used during training or hyperparameter tuning. This gives us a final evaluation that is closer to unseen data.


In [9]:
# RobustScaler is used to scale features in a way that is resistant to outliers.
# It is fitted ONLY on the training set to avoid data leakage from test data.

# Initialize the scaler
scaler = RobustScaler()

# Create copies of the training and test sets to avoid modifying original data
X_train_t = X_train.copy()
X_test_t = X_test.copy()

# Define the columns to be scaled (time and transaction amount)
scale_cols = ["Time", "Amount"]

# Fit the scaler on training data and transform training set
X_train_t[scale_cols] = scaler.fit_transform(X_train[scale_cols])

# Apply the same transformation to the test set (no refitting)
X_test_t[scale_cols] = scaler.transform(X_test[scale_cols])

# Display dataset shapes after scaling
print("Train shape:", X_train_t.shape, "| Test shape:", X_test_t.shape)

# Check scaled range for Time feature (training set)
print(
    "Scaled Time   -> min:", round(X_train_t["Time"].min(), 2),
    " max:", round(X_train_t["Time"].max(), 2),
)

# Check scaled range for Amount feature (training set)
print(
    "Scaled Amount -> min:", round(X_train_t["Amount"].min(), 2),
    " max:", round(X_train_t["Amount"].max(), 2),
)

Train shape: (227845, 30) | Test shape: (56962, 30)
Scaled Time   -> min: -1.0  max: 1.03
Scaled Amount -> min: -0.31  max: 357.26


### Why we scale only `Time` and `Amount`

The variables `V1` to `V28` are already PCA-transformed variables. `Time` and `Amount`, instead, are directly interpretable and have their own original scale.

We use `RobustScaler` because it scales using the median and interquartile range. This is suitable for `Amount`, which contains very large values and is not normally distributed.

The scaler is fitted on the training set only and then applied to the test set. This avoids data leakage from the test data into preprocessing.


## 6. Modelling strategy

At this point we have enough information to decide how to model the problem.

We compare three supervised classification model families:

1. **Logistic Regression**: a simple linear baseline. It helps us understand how far a transparent model can go.
2. **Random Forest**: an ensemble of decision trees. It can capture non-linear interactions and is usually strong on tabular data.
3. **XGBoost**: a boosted-tree model. It builds trees sequentially and often performs well on structured datasets, especially when the target is difficult.

The goal is not only to find the highest number in a table. We want to understand whether model complexity improves the ability to identify fraud and whether tuning actually helps.


## Evaluation strategy

Before comparing different models, we define a common evaluation function.
This is useful because every classifier will return fraud probabilities, and we want to evaluate all models using the same metrics and the same decision threshold.

The function converts predicted probabilities into class predictions using a threshold of 0.5, then computes ROC-AUC, PR-AUC, precision, recall, and F1-score.
We also store each result in a shared list, so that the models can be compared later in a final leaderboard.

Finally, we define a stratified cross-validation strategy. Stratification is important because the dataset is highly imbalanced, so each fold should preserve approximately the same proportion of legitimate and fraudulent transactions.


In [ ]:
MODEL_RESULTS = []


def evaluate_model(name, y_true, proba, threshold=0.5):
    """Evaluate a probabilistic binary classifier on the test set."""
    y_pred = (proba >= threshold).astype(int) #convert probabilities to class predictions
    row = {
        "name": name,
        "roc_auc": roc_auc_score(y_true, proba), #area under the ROC curve
        "pr_auc": average_precision_score(y_true, proba), #area under the PR curve
        "precision@0.5": precision_score(y_true, y_pred, zero_division=0), 
        "recall@0.5": recall_score(y_true, y_pred, zero_division=0), 
        "f1@0.5": f1_score(y_true, y_pred, zero_division=0),
    }
    print(pd.Series(row).to_string())
    MODEL_RESULTS.append(row) 
    return row


# Two folds keep the search light on a large dataset.
# With more time/resources, this could be increased to 3 or 5 folds.
CV = StratifiedKFold(n_splits=2, shuffle=True, random_state=RANDOM_STATE)


At this point, we have a consistent evaluation setup. This allows us to train different models and compare them fairly using the same metrics and validation logic.

## 6.1 Logistic Regression

We start with Logistic Regression because it is a good baseline model for binary classification.

Before running it, the expectation is clear: since it learns a mostly linear decision rule, it may not capture all fraud patterns. Still, it is useful because if a simple model already performs well, more complex models must justify their extra complexity.


In [ ]:
lr_default = LogisticRegression( 
    max_iter=5000, #max n of iterations
    random_state=RANDOM_STATE, #ensure reproducibility
    solver="saga", 
)
lr_default.fit(X_train_t, y_train)

p_lr_default = lr_default.predict_proba(X_test_t)[:, 1] #predict the probability of fraud for each transaction
print("=== LogisticRegression (default) — test ===")
row_lr_default = evaluate_model("LR default", y_test, p_lr_default)


=== LogisticRegression (default) — test ===
name             LR default
roc_auc            0.960782
pr_auc             0.743846
precision@0.5      0.828947
recall@0.5         0.642857
f1@0.5             0.724138


### Logistic Regression baseline result

The default Logistic Regression reaches **ROC-AUC = 0.961** and **PR-AUC = 0.744**.

ROC-AUC looks high, but PR-AUC is more important here because fraud is rare. At the default threshold of 0.5, recall is about **0.643**, which means the model misses a relevant share of fraud cases.

This gives us a first answer: a simple linear classifier can detect some structure, but it is probably not enough for the final model.


After testing Logistic Regression with its default settings, we now try to improve the model by tuning some hyperparameters.

To do this, we define a small set of possible values that the model will test in different combinations. These values were selected after a brief check of the most commonly tuned parameters for Logistic Regression in imbalanced binary classification problems.

The parameters we tune are:

* `C`: controls the regularization strength. A smaller value makes the model simpler and more regularized, while a larger value allows the model to fit the training data more closely.
* `class_weight`: controls whether the model should treat both classes equally or give more importance to the minority class. This is relevant because fraud cases are much rarer than legitimate transactions.

Since the search space is small, we use Grid Search. This means that the model will train and evaluate all possible combinations of these parameters using cross-validation, then keep the combination that gives the best average precision score.


In [ ]:
param_lr = {
    "C": [0.1, 1.0, 10.0], #controls the regularization strength. A smaller value makes the model simpler and more regularized, while a larger value allows the model to fit the training data more closely.
    "class_weight": [None, "balanced"], #controls whether the model should treat both classes equally or give more importance to minority class.
}

g_lr = GridSearchCV(
    LogisticRegression(max_iter=4000, random_state=RANDOM_STATE, solver="saga"), 
    param_grid=param_lr, #grid search to find the best hyperparameters
    scoring="average_precision", #we use average precision because fraud is rare
    cv=CV, 
    n_jobs=-1, #all available cores
    refit=True, 
)
g_lr.fit(X_train_t, y_train) 

print("Best params:", g_lr.best_params_) 
print("Best CV average_precision:", round(g_lr.best_score_, 6)) #best average precision score

p_lr_tuned = g_lr.predict_proba(X_test_t)[:, 1] 
print("\n=== LogisticRegression (tuned) — test ===") 
row_lr_tuned = evaluate_model("LR tuned", y_test, p_lr_tuned)


/Users/matildedovizio/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/matildedovizio/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/matildedovizio/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/matildedovizio/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/matildedovizio/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.w

Best params: {'C': 0.1, 'class_weight': None}
Best CV average_precision: 0.751486

=== LogisticRegression (tuned) — test ===
name             LR tuned
roc_auc          0.964021
pr_auc           0.742047
precision@0.5    0.828947
recall@0.5       0.642857
f1@0.5           0.724138


### Logistic Regression tuning result

The grid search selects **`C = 0.1`** and **`class_weight = None`**.

The tuned model slightly improves ROC-AUC, but PR-AUC decreases a little compared with the default Logistic Regression. This suggests that the main limitation is not only the selected hyperparameters. The linear form of the model itself is probably too simple for this dataset.


## 6.2 Random Forest

The next model is Random Forest. This is a natural second step because it keeps the problem in the family of classical machine learning models, but it is much more flexible than Logistic Regression.

While Logistic Regression learns a single linear decision rule, Random Forest can capture more complex and non-linear relationships between the transaction features and the target variable.

A Random Forest combines many decision trees. Each individual tree may be unstable and sensitive to noise, but the ensemble is usually more robust because the final prediction is aggregated across many trees.

In this first version, we do not perform hyperparameter tuning yet. Instead, we manually choose a reasonable starting configuration, including the number of trees, the maximum depth of each tree, and the minimum number of samples required in each leaf. This gives us a baseline Random Forest result that we can later compare with the tuned version.


In [ ]:
rf_default = RandomForestClassifier(
    n_estimators=200,#builds 200 decision trees
    max_depth=12, #each tree can grow up to depth 12. This limits complexity and helps avoid overfitting
    min_samples_leaf=2, 
    random_state=RANDOM_STATE, #fixes randomness so results are reproducible.
    n_jobs=-1,
)
rf_default.fit(X_train_t, y_train) 

p_rf_default = rf_default.predict_proba(X_test_t)[:, 1] #predict the probability of fraud for each trasaction 
print("=== RandomForest (default-ish) — test ===")
row_rf_default = evaluate_model("RF default", y_test, p_rf_default) #evaluate the model 


=== RandomForest (default-ish) — test ===
name             RF default
roc_auc            0.973583
pr_auc             0.868440
precision@0.5      0.941860
recall@0.5         0.826531
f1@0.5             0.880435


### Random Forest baseline result

The Random Forest baseline clearly improves over Logistic Regression. PR-AUC increases to **0.868**, and at threshold 0.5 the model reaches recall around **0.827** and precision around **0.942**.

This result is important because it shows that non-linear interactions between variables are useful in this dataset. The gain does not come from changing the metric; it comes from using a model family that can capture more complex patterns.


After the baseline Random Forest, we try to improve the model by tuning some of its main hyperparameters.

We define a set of possible values for the parameters that are usually important in Random Forest models:

* `n_estimators`: the number of trees in the forest. More trees can make the model more stable, but also slower.
* `max_depth`: the maximum depth of each tree. Deeper trees can capture more complex patterns, but they can also overfit.
* `min_samples_leaf`: the minimum number of samples required in a final leaf node. Higher values make the trees more conservative and can reduce overfitting.
* `class_weight`: controls whether the model should give more importance to the minority class, which is relevant because fraudulent transactions are much rarer than legitimate ones.

Since the full search space contains many possible combinations, we use `RandomizedSearchCV`. This means that only 8 randomly selected combinations are trained and evaluated using cross-validation. The best combination is selected according to average precision, which is a suitable metric for imbalanced fraud detection.


In [ ]:
param_rf = {
    "n_estimators": [200, 400], 
    "max_depth": [8, 12, 16, None], #none means that the tree can grow as deep as needed to reduce impurity
    "min_samples_leaf": [1, 2, 4], #highest values make the model more conservative and reduce overfitting
    "class_weight": [None, "balanced"],
}

r_rf = RandomizedSearchCV(
    RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
    param_distributions=param_rf,
    n_iter=8, #randomly selects 8 combinations from the grid
    scoring="average_precision", #we use average precision because fraud is rare
    cv=CV,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    refit=True,
)
r_rf.fit(X_train_t, y_train)

print("Best params:", r_rf.best_params_)
print("Best CV average_precision:", round(r_rf.best_score_, 6))

p_rf_tuned = r_rf.predict_proba(X_test_t)[:, 1] #the best combination is used to predict the probability of fraud for each transaction
print("\n=== RandomForest (tuned) — test ===")
row_rf_tuned = evaluate_model("RF tuned", y_test, p_rf_tuned)


Best params: {'n_estimators': 200, 'min_samples_leaf': 1, 'max_depth': 16, 'class_weight': None}
Best CV average_precision: 0.840098

=== RandomForest (tuned) — test ===
name             RF tuned
roc_auc          0.978570
pr_auc           0.872194
precision@0.5    0.941860
recall@0.5       0.826531
f1@0.5           0.880435


### Random Forest tuning result

The randomized search selects **200 trees**, **max_depth = 16**, **min_samples_leaf = 1**, and **no class weighting**.

The tuned Random Forest improves PR-AUC only slightly, from **0.868** to **0.872**. Precision, recall, and F1 at the default threshold remain effectively unchanged.

So the tuning helps, but only marginally. Most of the improvement already came from moving from a linear model to a tree-based ensemble.


## 6.3 XGBoost

The last model family is XGBoost. Unlike Random Forest, which trains many trees mostly independently, XGBoost builds trees sequentially. Each new tree focuses on errors left by previous trees.

This is useful for a difficult classification problem because the model can focus more on hard-to-classify transactions. We also compute `scale_pos_weight` from the training data to reflect the imbalance between legitimate and fraudulent transactions.


In [ ]:
pos = int((y_train == 1).sum()) #n of fraud cases
neg = int((y_train == 0).sum()) #n of legitimate cases
scale_pos_weight = max(neg / max(pos, 1), 1.0) #to balance the classes. legitimate>fraud

xgb_default = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.08,#controls how much each tree contributes to the final predictions
    subsample=0.8,
    colsample_bytree=0.8,#reduce overfitting
    reg_lambda=1.0, #controls the regularizaion strenght
    random_state=RANDOM_STATE,
    eval_metric="logloss", 
    scale_pos_weight=scale_pos_weight,
    n_jobs=-1, 
    tree_method="hist", #use histogram-based tree method
)
xgb_default.fit(X_train_t, y_train)

p_xgb_default = xgb_default.predict_proba(X_test_t)[:, 1]
print("=== XGBoost (default-ish) — test ===")
row_xgb_default = evaluate_model("XGB default", y_test, p_xgb_default)


=== XGBoost (default-ish) — test ===
name             XGB default
roc_auc             0.979504
pr_auc              0.867820
precision@0.5       0.777778
recall@0.5          0.857143
f1@0.5              0.815534


### XGBoost baseline result

The first XGBoost model reaches **ROC-AUC = 0.980** and **PR-AUC = 0.868**, which is close to the Random Forest baseline.

At threshold 0.5, it has higher recall than the Random Forest but lower precision. In practical terms, it catches slightly more frauds, but it also creates more false alarms.

This shows why looking at one metric only is risky. Depending on the business cost of false positives and false negatives, the preferred model may change.


In [ ]:
param_xgb = {
    # Number of boosting rounds: keep around the best area, without making it too heavy
    "n_estimators": [500, 600, 700, 800],

    # Tree complexity: medium-depth trees usually work well for tabular fraud data
    "max_depth": [4, 5, 6],

    # Learning speed: smaller values are safer, but require enough estimators
    "learning_rate": [0.04, 0.05, 0.06, 0.07],

    # Row sampling: helps reduce overfitting
    "subsample": [0.7, 0.75, 0.8, 0.85],

    # Feature sampling: the previous result preferred high values
    "colsample_bytree": [0.9, 0.95, 1.0],

    # Regularization: previous result selected a high lambda, so we test around it
    "reg_lambda": [6.0, 8.0, 10.0, 12.0],

    # L1 regularization: small values can help control complexity
    "reg_alpha": [0.05, 0.1, 0.2],

    # Controls how conservative the tree splits are
    "min_child_weight": [3, 4, 5, 6],

    # Minimum loss reduction required to split a node
    "gamma": [0.05, 0.1, 0.15],

    # Useful for imbalanced logistic classification
    "max_delta_step": [1, 2, 3],

    # Since the previous best used a lower class weight, we explore lower values
    "scale_pos_weight": [
        scale_pos_weight * 0.4,
        scale_pos_weight * 0.5,
        scale_pos_weight * 0.6,
        scale_pos_weight * 0.7,
        scale_pos_weight * 0.8,
    ],
}

r_xgb = RandomizedSearchCV(
    estimator=XGBClassifier(
        random_state=RANDOM_STATE, 
        objective="binary:logistic", 
        eval_metric="aucpr",
        n_jobs=-1,
        tree_method="hist",
    ),
    param_distributions=param_xgb, #randomly selects 50 combinations from the grid
    n_iter=50,
    scoring="average_precision",
    cv=CV, 
    random_state=RANDOM_STATE + 2, 
    n_jobs=-1,
    refit=True,
    verbose=0,
)

r_xgb.fit(X_train_t, y_train)

print("Best params:", r_xgb.best_params_)
print("Best CV average_precision:", round(r_xgb.best_score_, 6)) 

p_xgb = r_xgb.predict_proba(X_test_t)[:, 1]

print("\n=== XGBoost (tuned) — test ===")
row_xgb = evaluate_model("XGB tuned", y_test, p_xgb)

Best params: {'subsample': 0.7, 'scale_pos_weight': 346.3720812182741, 'reg_lambda': 6.0, 'reg_alpha': 0.2, 'n_estimators': 800, 'min_child_weight': 3, 'max_depth': 4, 'max_delta_step': 2, 'learning_rate': 0.04, 'gamma': 0.1, 'colsample_bytree': 0.95}
Best CV average_precision: 0.846

=== XGBoost (tuned) — test ===
name             XGB tuned
roc_auc           0.980988
pr_auc            0.877395
precision@0.5     0.828283
recall@0.5        0.836735
f1@0.5            0.832487


### XGBoost second tuning result

The second tuning pass gives the highest PR-AUC in the notebook: **0.8799**.

The difference from the first tuned XGBoost is very small. The second pass slightly increases recall but slightly lowers precision and F1. This means the two tuned XGBoost versions are practically close, and the final decision depends on whether the priority is catching a few more frauds or reducing false alarms.


## 7. Model comparison

Now we compare all trained models on the same test set.

The table is sorted by **PR-AUC** because, after inspecting the class distribution, we know that this is a rare positive-class problem. PR-AUC is therefore more informative than plain accuracy.


In [ ]:
summary = pd.DataFrame(MODEL_RESULTS) #create a dataframe with the results
summary = summary.sort_values("pr_auc", ascending=False).reset_index(drop=True) #sort it by PR-AUC

display(summary) 
print("\nSorted by pr_auc (higher is better for finding fraud under imbalance).") 


,name,roc_auc,pr_auc,precision@0.5,recall@0.5,f1@0.5
0,XGB tuned,0.980988,0.877395,0.828283,0.836735,0.832487
1,RF tuned,0.978570,0.872194,0.941860,0.826531,0.880435
2,RF default,0.973583,0.868440,0.941860,0.826531,0.880435
3,XGB default,0.979504,0.867820,0.777778,0.857143,0.815534
4,LR default,0.960782,0.743846,0.828947,0.642857,0.724138
5,LR tuned,0.964021,0.742047,0.828947,0.642857,0.724138



Sorted by pr_auc (higher is better for finding fraud under imbalance).


### Reading the leaderboard

The best PR-AUC is obtained by **XGB tuned pass2** (**0.8799**), but **XGB tuned** is almost identical (**0.8793**). The tuned Random Forest is slightly lower in PR-AUC (**0.8722**), but it has the strongest F1-score at the default 0.5 threshold.

Therefore, the conclusion is not simply “one model is always best”. The decision depends on the operational objective:

- choose **XGB tuned pass2** if the priority is the best ranking performance and slightly higher recall;
- choose **RF tuned** if the priority is a strong default-threshold balance with higher precision and F1;
- choose **XGB tuned** if the goal is a strong compromise between PR-AUC, recall, and precision.


In [ ]:
results_df = pd.DataFrame(MODEL_RESULTS) 

ranking = results_df.melt( #reshape the dataframe to long format
    id_vars="name",
    value_vars=["pr_auc", "roc_auc"],
    var_name="metric", 
    value_name="score",
)

fig1 = px.bar( #create a bar chart
    ranking.sort_values("score"), #sort the values by score
    x="score", 
    y="name",
    color="metric",
    orientation="h", #horizontal bar chart
    barmode="group",
    range_x=[0, 1],
    text_auto=".3f", #show the score on the chart
    title="Model comparison — PR-AUC and ROC-AUC",
)
fig1.update_layout(xaxis_title="Score", yaxis_title="", legend_title="") 
fig1.show()

operating = results_df.melt( 
    id_vars="name",
    value_vars=["precision@0.5", "recall@0.5", "f1@0.5"], #metrics to compare
    var_name="metric",
    value_name="score",
)

fig2 = px.bar( 
    operating.sort_values("score"),
    x="score",
    y="name",
    color="metric",
    orientation="h",
    barmode="group",
    range_x=[0, 1],
    text_auto=".2f",
    title="Model comparison — precision, recall and F1 at threshold 0.5",
)
fig2.update_layout(xaxis_title="Score", yaxis_title="", legend_title="")
fig2.show()


## 8. Did tuning help?

After comparing the final scores, we also check whether hyperparameter tuning actually improved each model family.

This matters because tuning costs time and computation. A more complex search is useful only if it produces a meaningful improvement.


In [ ]:
results_df = pd.DataFrame(MODEL_RESULTS)
METRICS = ["pr_auc", "roc_auc", "precision@0.5", "recall@0.5", "f1@0.5"] #metrics to compare


def model_family(name): #function to identify the model family
    if name.startswith("LR"): #if name starts with LR= Logistic regression
        return "Logistic regression"
    if name.startswith("RF"): #if name starts with RF= Random Forest
        return "Random forest"
    if name.startswith("XGB"): #if name starts with XGB= XGBoost
        return "XGBoost"
    return name


def model_version(name): #function to identify the model version
    return "default" if "default" in name.lower() else "tuned" #if name contains default= default, otherwise tuned


comparison = results_df.copy() 
comparison["family"] = comparison["name"].map(model_family) 
comparison["version"] = comparison["name"].map(model_version)

best_default = (
    comparison[comparison["version"] == "default"] #select the default version
    .sort_values("pr_auc", ascending=False)
    .groupby("family", as_index=False)
    .first()
)

best_tuned = (
    comparison[comparison["version"] == "tuned"] #select the tuned veriosn
    .sort_values("pr_auc", ascending=False)
    .groupby("family", as_index=False)
    .first()
)

pair = pd.concat([best_default, best_tuned], ignore_index=True) #concatenate the default and tuned versions

wide = pair.pivot_table(index="family", columns="version", values=METRICS) #pivot the dataframe to wide format
columns = {}
for metric in METRICS: #dictionary to store the metrics
    columns[(metric, "default")] = wide[(metric, "default")]
    columns[(metric, "tuned")] = wide[(metric, "tuned")]
    columns[(metric, "delta (tuned-default)")] = (
        wide[(metric, "tuned")] - wide[(metric, "default")]
    )

paired = pd.DataFrame(columns).round(4) #dataframe with the metrics
paired.columns = pd.MultiIndex.from_tuples(paired.columns) #create a multi-level index

print("Default vs best-tuned per model (delta = tuned - default; which tuned run was kept):") # default vs best-tuned per model
print("kept tuned runs:", dict(zip(best_tuned["family"], best_tuned["name"])))
display(paired)

long = pair.melt( 
    id_vars=["family", "version"],
    value_vars=METRICS,
    var_name="metric",
    value_name="score",
)

fig = px.bar( 
    long,
    x="metric",
    y="score",
    color="version",
    barmode="group",
    facet_col="family",
    text_auto=".3f",
    range_y=[0, 1],
    category_orders={"version": ["default", "tuned"], "metric": METRICS},
    title="Default vs tuned — every metric, per model family",
)
fig.update_layout(legend_title="", yaxis_title="Score", xaxis_title="") 
fig.for_each_xaxis(lambda axis: axis.update(tickangle=45)) 
fig.for_each_annotation(lambda annotation: annotation.update(text=annotation.text.split("=")[-1]))
fig.show()


Default vs best-tuned per model (delta = tuned - default; which tuned run was kept):
kept tuned runs: {'Logistic regression': 'LR tuned', 'Random forest': 'RF tuned', 'XGBoost': 'XGB tuned'}


pr_auc                                 roc_auc           \
                     default    tuned delta (tuned-default)  default    tuned   
family                                                                          
Logistic regression 0.743800 0.742000             -0.001800 0.960800 0.964000   
Random forest       0.868400 0.872200              0.003800 0.973600 0.978600   
XGBoost             0.867800 0.877400              0.009600 0.979500 0.981000   

                                          precision@0.5           \
                    delta (tuned-default)       default    tuned   
family                                                             
Logistic regression              0.003200      0.828900 0.828900   
Random forest                    0.005000      0.941900 0.941900   
XGBoost                          0.001500      0.777800 0.828300   

                                          recall@0.5           \
                    delta (tuned-default)    default    tuned   
family                                                          
Logistic regression              0.000000   0.642900 0.642900   
Random forest                    0.000000   0.826500 0.826500   
XGBoost                          0.050500   0.857100 0.836700   

                                            f1@0.5           \
                    delta (tuned-default)  default    tuned   
family                                                        
Logistic regression              0.000000 0.724100 0.724100   
Random forest                    0.000000 0.880400 0.880400   
XGBoost                         -0.020400 0.815500 0.832500   

                                           
                    delta (tuned-default)  
family                                     
Logistic regression              0.000000  
Random forest                    0.000000  
XGBoost                          0.017000

### Tuning interpretation

Tuning does not help all models in the same way.

- **Logistic Regression:** tuning does not improve PR-AUC. This supports the idea that the linear model is structurally limited for this task.
- **Random Forest:** tuning gives a small PR-AUC improvement, but the operating metrics at threshold 0.5 remain unchanged.
- **XGBoost:** tuning gives the clearest improvement, especially in PR-AUC and F1 compared with the default XGBoost model.

So tuning is useful, but the larger lesson is model selection: moving from a simple linear model to tree-based ensembles had a bigger impact than fine-tuning alone.


## 9. Final conclusion

This notebook follows a complete supervised learning workflow for a fraud-detection problem.

We began from the dataset, not from the final model. First we checked that the data was loaded correctly, verified the target column, inspected the class labels, counted missing values, and measured the imbalance. Only after seeing that fraud represents about **0.173%** of transactions did we decide that accuracy was not enough.

The exploratory analysis showed that `Amount` is skewed and that some anonymised PCA variables have stronger linear association with the target than others. This justified a controlled preprocessing step: separating features and target, using a stratified train/test split, and scaling only `Time` and `Amount` with a scaler fitted on the training set.

The modelling comparison shows a clear pattern. Logistic Regression provides a useful baseline, but tree-based ensemble models perform better. Random Forest is very strong at the default 0.5 threshold, especially in precision and F1. XGBoost, after tuning, gives the best PR-AUC and therefore the best ranking performance for the rare fraud class.

The final recommendation is not based on accuracy. If the main objective is to rank suspicious transactions for review, **XGB tuned pass2** is the strongest candidate because it obtains the highest PR-AUC. If the process needs a model that performs very well at the default threshold with fewer false alarms, **Random Forest tuned** is also a credible choice.

From a business perspective, the main point is that false negatives and false positives do not have the same cost. Missing a fraud can create direct financial loss, while false alarms create investigation costs and customer friction. The best model should therefore be chosen together with a threshold policy, not only by looking at a single test-set score.


## 10. Limitations and possible improvements

This project is intentionally kept close to the course workflow and to the logic of a readable student notebook. Some possible improvements are:

- evaluate multiple probability thresholds instead of relying only on 0.5;
- add confusion matrices for the final selected models;
- estimate the business cost of false positives and false negatives more explicitly;
- test the effect of removing duplicated rows;
- use cross-validation more extensively if more computation time is available;
- analyze errors by inspecting the transactions that are repeatedly misclassified;
- compare the models using a precision-recall curve instead of only scalar metrics.

The main limitation is that the original variables `V1` to `V28` are anonymised PCA components. This protects privacy, but it makes business interpretation harder. For this reason, the notebook focuses more on predictive performance and methodological correctness than on explaining individual feature meanings.
